# Tool Usage Diversity

The Tool Usage Diversity notebook contains the implementation, tests, and usage workflow for the Tool Usage Diversity metric. It scans JSON/JSONL conversation traces, counts structured assistant tool calls, and calculates normalized entropy against one fixed tool inventory.

The score is `H / ln(K)`, where `K` is the number of configured tools. A reference tool with no calls remains in the denominator. No recognized calls means the score is `None` (undefined). Unknown tools are reported and excluded from scoring. A one-tool inventory scores `0.0` when it has calls. The score measures the aggregate tool-call distribution, not equality of usage on every task.

The parser handles structured `tool_calls` and legacy `function_call` fields on assistant/AI messages. It does not infer calls from text or tool results. The supplied tau2 `results.json` layout (`simulations` with `messages`) has been checked; other benchmark output formats may need a parser adjustment.

To run: open this notebook, run the cells from top to bottom, edit the configuration in the final cell, then run that cell. The tests use only Python's standard library. You will need to have ran tau-bench as well as know the paths to the input and output of your results.

In [ ]:
from collections import Counter
from pathlib import Path
from typing import Any
import json
import math


def tool_name(call: Any) -> str | None:
    """Get a tool name from a common structured tool-call object."""
    if not isinstance(call, dict):
        return None
    name = call.get("name")
    if not isinstance(name, str) and isinstance(call.get("function"), dict):
        name = call["function"].get("name")
    return name.strip() if isinstance(name, str) and name.strip() else None


def extract_tool_calls(record: Any) -> list[str]:
    """Extract structured calls from one message list or conversation record."""
    if isinstance(record, list):
        messages = record
    elif isinstance(record, dict):
        messages = next(
            (record[key] for key in ("messages", "conversation", "trajectory")
             if isinstance(record.get(key), list)),
            None,
        )
        if messages is None and record.get("role", record.get("type")) in {
            "assistant", "ai", "AIMessage"
        }:
            messages = [record]
    else:
        return []

    if not isinstance(messages, list):
        return []

    calls, seen_ids = [], set()
    for message in messages:
        if not isinstance(message, dict):
            continue
        payload = message.get("data", message)
        if not isinstance(payload, dict):
            continue
        role = str(message.get("role", message.get("type", ""))).lower()
        if role not in {"assistant", "ai", "aimessage"}:
            continue

        raw_calls = payload.get("tool_calls")
        if not isinstance(raw_calls, list):
            extra = payload.get("additional_kwargs", {})
            raw_calls = extra.get("tool_calls") if isinstance(extra, dict) else None
        if not isinstance(raw_calls, list):
            legacy = payload.get("function_call")
            raw_calls = [legacy] if isinstance(legacy, dict) else []

        for call in raw_calls:
            name = tool_name(call)
            if name is None:
                continue
            call_id = call.get("id") if isinstance(call, dict) else None
            if call_id and call_id in seen_ids:
                continue
            if call_id:
                seen_ids.add(call_id)
            calls.append(name)
    return calls


def calculate_diversity(call_counts: dict[str, int], reference_tools: list[str]) -> dict[str, Any]:
    """Calculate entropy and report counts for a fixed reference inventory."""
    tools = sorted(set(reference_tools))
    if not tools:
        raise ValueError("Configure at least one reference tool.")
    if any(not isinstance(n, int) or isinstance(n, bool) or n < 0 for n in call_counts.values()):
        raise ValueError("Tool counts must be non-negative integers.")

    counts = {name: call_counts.get(name, 0) for name in tools}
    unknown = {name: count for name, count in sorted(call_counts.items())
               if name not in tools and count > 0}
    total = sum(counts.values())
    entropy = -sum((count / total) * math.log(count / total)
                   for count in counts.values() if count) if total else None
    score = (0.0 if len(tools) == 1 else max(0.0, min(1.0, entropy / math.log(len(tools))))) if total else None
    used = sum(count > 0 for count in counts.values())
    balance = (0.0 if used <= 1 else entropy / math.log(used)) if total else None

    return {
        "total_tool_calls": total + sum(unknown.values()),
        "recognized_tool_calls": total,
        "unknown_tool_calls": sum(unknown.values()),
        "unknown_tool_call_counts": unknown,
        "distinct_tools_used": used,
        "reference_tool_count": len(tools),
        "tool_call_counts": counts,
        "tool_call_proportions": {name: count / total if total else 0.0
                                   for name, count in counts.items()},
        "raw_shannon_entropy": entropy,
        "tool_balance": balance,
        "normalized_tool_usage_diversity": score,
    }




In [ ]:
def read_records(path: Path) -> list[Any]:
    """Read JSON records or JSONL messages/tasks from a file."""
    if path.suffix.lower() == ".json":
        value = json.loads(path.read_text(encoding="utf-8-sig"))
        if isinstance(value, dict):
            simulations = value.get("simulations")
            if isinstance(simulations, list):
                info = value.get("info", {})
                environment = info.get("environment_info", {}) if isinstance(info, dict) else {}
                source_domain = environment.get("domain_name") if isinstance(environment, dict) else None
                return [dict(row, _source_domain=source_domain)
                        for row in simulations if isinstance(row, dict)]
            return value.get("tasks", [value])
        if isinstance(value, list):
            return [value] if value and all(isinstance(row, dict) and "role" in row for row in value) else value
        return []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines()
            if line.strip()]
    if rows and all(isinstance(row, dict) and ("role" in row or "type" in row) for row in rows):
        return [rows]
    return [record for row in rows
            for record in (row.get("tasks", [row]) if isinstance(row, dict) else [row])]


def analyze_dataset(
    input_paths: list[Path],
    reference_tools: list[str],
    task_ids: set[str] | None = None,
    domain: str | None = None,
) -> dict[str, Any]:
    """Scan JSON/JSONL inputs, count calls, and return metrics and diagnostics."""
    files = set()
    for path in input_paths:
        if not path.exists():
            raise FileNotFoundError(f"Input does not exist: {path}")
        if path.is_file():
            if path.suffix.lower() not in {".json", ".jsonl"}:
                raise ValueError(f"Unsupported input file type: {path}")
            files.add(path.resolve())
        else:
            files.update(item.resolve() for item in path.rglob("*")
                         if item.is_file() and item.suffix.lower() in {".json", ".jsonl"})

    counts, task_calls, warnings = Counter(), [], []
    parsed_files = skipped_files = conversation_files = task_records = 0
    if not files:
        warnings.append("No JSON or JSONL input files found.")

    for path in sorted(files, key=lambda item: str(item).casefold()):
        try:
            records = read_records(path)
        except (OSError, UnicodeError, json.JSONDecodeError, ValueError) as error:
            warnings.append(f"Skipped malformed file {path}: {error}")
            skipped_files += 1
            continue
        parsed_files += 1
        records_in_file = 0
        for index, record in enumerate(records):
            if isinstance(record, dict):
                task_id = record.get("task_id", record.get("id"))
                metadata = record.get("metadata", {})
                record_domain = record.get("domain", record.get("_source_domain",
                    metadata.get("domain") if isinstance(metadata, dict) else None))
            else:
                task_id = record_domain = None
            if task_ids and str(task_id) not in task_ids:
                continue
            if domain and str(record_domain or "").casefold() != domain.casefold():
                continue
            calls = extract_tool_calls(record)
            if not isinstance(record, (dict, list)):
                continue
            task_records += 1
            records_in_file += 1
            counts.update(calls)
            task_calls.append({"task_id": str(task_id or f"{path.name}#{index + 1}"), "calls": calls})
        if records_in_file:
            conversation_files += 1
        else:
            skipped_files += 1
            warnings.append(f"Skipped {path}: no supported conversation records matched.")

    report = {
        "input_paths": [str(path.resolve()) for path in input_paths],
        "discovered_files": [str(path) for path in sorted(files, key=lambda item: str(item).casefold())],
        "files_processed": len(files),
        "parsed_files": parsed_files,
        "skipped_files": skipped_files,
        "conversation_files_processed": conversation_files,
        "task_records_processed": task_records,
        "reference_tools": sorted(set(reference_tools)),
        "warnings": warnings,
        "task_calls": task_calls,
        **calculate_diversity(counts, reference_tools),
    }
    if report["unknown_tool_calls"]:
        report["warnings"].append("Unknown tool calls are reported but excluded from the score.")
    return report
def validate_perturbations(report: dict[str, Any], reference_tools: list[str]) -> dict[str, Any]:
    """Compare baseline with global-single-tool and per-task-single-tool subsets."""
    calls = [name for task in report["task_calls"] for name in task["calls"]
             if name in reference_tools]
    baseline = report["normalized_tool_usage_diversity"]

    def compare(selected_calls: list[str]) -> dict[str, Any]:
        metrics = calculate_diversity(dict(Counter(selected_calls)), reference_tools)
        score = metrics["normalized_tool_usage_diversity"]
        return {
            "tool_call_count": metrics["recognized_tool_calls"],
            "tools_represented": sorted(set(selected_calls)),
            "normalized_tool_usage_diversity": score,
            "absolute_score_change": abs(baseline - score) if baseline is not None and score is not None else None,
            "relative_score_change": (score - baseline) / baseline
                if baseline not in (None, 0) and score is not None else None,
            "expected_decrease": baseline > score + 1e-12
                if baseline is not None and score is not None else None,
        }

    selected = max(Counter(calls), key=lambda name: (Counter(calls)[name], name)) if calls else None
    strict = compare([name for name in calls if name == selected]) if selected else compare([])
    strict.update({"selected_tool": selected, "passed": strict["normalized_tool_usage_diversity"] == 0.0
                   if selected else None})
    single_tool_tasks = [task["calls"] for task in report["task_calls"]
                         if len(set(task["calls"]) & set(reference_tools)) == 1]
    task_level = compare([name for task in single_tool_tasks for name in task
                          if name in reference_tools])
    task_level.update({"task_count": len(single_tool_tasks), "supported": bool(single_tool_tasks)})
    return {"baseline_diversity": baseline, "baseline_tools_represented": sorted(set(calls)),
            "strict_single_tool": strict, "task_level_single_tool_subset": task_level}


def print_summary(report: dict[str, Any]) -> None:
    """Print a concise summary alongside the full JSON report."""
    score = report["normalized_tool_usage_diversity"]
    print(f"Tool Usage Diversity: {'undefined' if score is None else f'{score:.6f}'}")
    print(f"Files: {report['parsed_files']} parsed, {report['skipped_files']} skipped")
    print(f"Tasks: {report['task_records_processed']}; calls: {report['recognized_tool_calls']} recognized, "
          f"{report['unknown_tool_calls']} unknown")
    for tool, count in report["tool_call_counts"].items():
        print(f"  {tool}: {count} ({report['tool_call_proportions'][tool]:.3f})")
    for warning in report["warnings"]:
        print(f"Warning: {warning}")

SyntaxError: invalid syntax (1204069125.py, line 142)